## **Upload the kaggle API File**

In [2]:
import os
from google.colab import files

# Kaggle key upload
print("Please upload `kaggle.json` file:")
files.upload()

# API Permission
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle Setup Complete!")

Please upload `kaggle.json` file:


Saving kaggle.json to kaggle.json
Kaggle Setup Complete!


## **Install Libraries**

In [1]:
import cv2
import os
import numpy as np
import shutil
import random
from PIL import Image

## **1.** **Attendance System Datasets Download**

In [ ]:
# 1. iiitm-face-emotion
if not os.path.exists('data_bits'):
    print("Downloading iiitm-face-emotion_Dataset...")
    !kaggle datasets download -d rrsdiat/iiitm-face-emotion
    !unzip -q iiitm-face-emotion.zip -d data_iiitm-face-emotion

# 2. opencv-facial-recognition-lbph
if not os.path.exists('data_bits'):
    print("Downloading opencv-facial-recognition-lbph_Dataset...")
    !kaggle datasets download -d juniorbueno/opencv-facial-recognition-lbph
    !unzip -q opencv-facial-recognition-lbph.zip -d data_opencv-facial

# 3. lfw simulated masked face Dataset (Masked face)
if not os.path.exists('data_bits'):
    print("Downloading lfw-simulated-masked-face-dataset...")
    !kaggle datasets download -d muhammeddalkran/lfw-simulated-masked-face-dataset
    !unzip -q lfw-simulated-masked-face-dataset.zip -d data_lfw-simulated

# 4. sof-dataset-for-face-recognition Dataset (Wear Spec)
if not os.path.exists('data_bits'):
    print("Downloading sof-dataset-for-face-recognition...")
    !kaggle datasets download -d abhranta/sof-dataset-for-face-recognition
    !unzip -q sof-dataset-for-face-recognition.zip -d data_sof

print("\n Datasets Download!")

Dataset URL: https://www.kaggle.com/datasets/rrsdiat/iiitm-face-emotion
License(s): unknown
 98% 363M/371M [00:00<00:00, 598MB/s]
100% 371M/371M [00:00<00:00, 655MB/s]
Dataset URL: https://www.kaggle.com/datasets/juniorbueno/opencv-facial-recognition-lbph
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
  0% 0.00/5.89M [00:00<?, ?B/s]
100% 5.89M/5.89M [00:00<00:00, 1.23GB/s]
Dataset URL: https://www.kaggle.com/datasets/muhammeddalkran/lfw-simulated-masked-face-dataset
License(s): copyright-authors
  0% 0.00/84.8M [00:00<?, ?B/s]
100% 84.8M/84.8M [00:00<00:00, 1.55GB/s]
Dataset URL: https://www.kaggle.com/datasets/abhranta/sof-dataset-for-face-recognition
License(s): unknown
  0% 0.00/57.3M [00:00<?, ?B/s]
100% 57.3M/57.3M [00:00<00:00, 1.51GB/s]

 Datasets Download!


## **2. Data Processing & Splitting Configuration**

In [ ]:
source_folders = [

    'data_opencv-facial',
    'data_lfw-simulated',
    'data_sof',
    'data_iiitm-face-emotion',

]

base_output_dir = "Final_Attendance_Dataset"
train_dir = os.path.join(base_output_dir, "train")
test_dir = os.path.join(base_output_dir, "test")

img_size = (200, 200) # LBPH
split_ratio = 0.8     # 80% Train, 20% Test
min_images = 4        # minimum image count

# Face Detector Loading
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# old folder remove and new folders Create
if os.path.exists(base_output_dir):
    shutil.rmtree(base_output_dir)
os.makedirs(train_dir)
os.makedirs(test_dir)

current_id = 1
person_map = {}

print("Processing Start")

for db_folder in source_folders:
    if not os.path.exists(db_folder):
        print(f"Skipping {db_folder} (Folder not found)")
        continue

    print(f"-> Processing Dataset: {db_folder}...")

    # Dataset adding images
    for root, dirs, files in os.walk(db_folder):
        image_files = [f for f in files if f.endswith(('.jpg', '.png', '.jpeg', '.gif', '.JPEG'))]

        if len(image_files) >= min_images:
            #  Get Person name (Folder path)
            person_name = os.path.basename(root)

            # Remove unnessary names
            if person_name.lower() in ['images', 'train', 'test', 'data', 'lfw', 'masked', 'SoF', 'AbdA', 'SUB', 'subject']:
                person_name = os.path.basename(os.path.dirname(root)) + "_" + person_name

            person_label = f"User_{current_id}"
            person_map[person_label] = person_name

            processed_faces = []

            for file in image_files:
                try:
                    img_path = os.path.join(root, file)
                    pil_image = Image.open(img_path).convert('L') # Grayscale
                    image_array = np.array(pil_image, 'uint8')

                    # Face Detect
                    faces = face_cascade.detectMultiScale(image_array, 1.2, 5)

                    for (x, y, w, h) in faces:
                        face_resized = cv2.resize(image_array[y:y+h, x:x+w], img_size)
                        processed_faces.append(face_resized)

                except Exception:
                    continue

            # Images filter
            if len(processed_faces) >= min_images:
                random.shuffle(processed_faces)
                split_point = int(len(processed_faces) * split_ratio)

                # Test folder not be empty
                if split_point == len(processed_faces):
                    split_point = len(processed_faces) - 1

                train_imgs = processed_faces[:split_point]
                test_imgs = processed_faces[split_point:]

                # Save to Train
                for i, img in enumerate(train_imgs):
                    filename = f"{person_label}_{i+1}.jpg"
                    cv2.imwrite(os.path.join(train_dir, filename), img)

                # Save to Test
                for i, img in enumerate(test_imgs):
                    filename = f"{person_label}_{i+1}.jpg"
                    cv2.imwrite(os.path.join(test_dir, filename), img)

                current_id += 1
                if current_id % 100 == 0:
                    print(f"   Processed {current_id} users...")

# Names List Save
with open(os.path.join(base_output_dir, "names.txt"), "w") as f:
    for pid, pname in person_map.items():
        f.write(f"{pid} : {pname}\n")

print("\n-------------------------------------------")
print("Process Successfully")
print(f"Total Persons: {current_id - 1}")
print(f"Training Photos: {len(os.listdir(train_dir))}")
print(f"Testing Photos: {len(os.listdir(test_dir))}")
print("-------------------------------------------")

Processing Start
-> Processing Dataset: data_opencv-facial...
-> Processing Dataset: data_lfw-simulated...
-> Processing Dataset: data_sof...
-> Processing Dataset: data_iiitm-face-emotion...
   Processed 100 users...

-------------------------------------------
Process Successfully
Total Persons: 121
Training Photos: 2284
Testing Photos: 626
-------------------------------------------


In [4]:
!unzip -q Final_Attendance_Dataset.zip -d Final_Attendance_Dataset #Unzip Upload Dataset

## **3. Dataset Zip Download**

In [ ]:
!zip -r Final_Attendance_Dataset.zip Final_Attendance_Dataset
from google.colab import files
files.download('Final_Attendance_Dataset.zip')

  adding: Final_Attendance_Dataset/ (stored 0%)
  adding: Final_Attendance_Dataset/names.txt (deflated 63%)
  adding: Final_Attendance_Dataset/test/ (stored 0%)
  adding: Final_Attendance_Dataset/test/User_32_6.jpg (deflated 0%)
  adding: Final_Attendance_Dataset/test/User_32_149.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_43_3.jpg (deflated 0%)
  adding: Final_Attendance_Dataset/test/User_5_6.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_55_3.jpg (deflated 0%)
  adding: Final_Attendance_Dataset/test/User_27_2.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_32_276.jpg (deflated 0%)
  adding: Final_Attendance_Dataset/test/User_75_2.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_116_1.jpg (deflated 0%)
  adding: Final_Attendance_Dataset/test/User_28_1.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_73_1.jpg (stored 0%)
  adding: Final_Attendance_Dataset/test/User_63_1.jpg (deflated 0%)
  adding: Final_Attendance_Data

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **4. Model Training (LBPH Algorithm)**

In [5]:
import cv2
import numpy as np
import os
from PIL import Image

# 1. Path Settings
dataset_path = 'Final_Attendance_Dataset/train' # Train folder
model_save_path = 'trainer.yml'                 # Model Name

# 2. Recognizer and Detector initialize
recognizer = cv2.face.LBPHFaceRecognizer_create()
detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def getImagesAndLabels(path):
    imagePaths = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(('.jpg', '.png'))]

    faceSamples = []
    ids = []

    print(f"Training Images {len(imagePaths)} Reading")

    for imagePath in imagePaths:
        try:
            # Image Grayscale
            PIL_img = Image.open(imagePath).convert('L')
            img_numpy = np.array(PIL_img, 'uint8')

            # File name  (ID sepearte)
            # Format: User_1_1.jpg -> ID = 1
            filename = os.path.split(imagePath)[-1]
            id = int(filename.split("_")[1])

            faceSamples.append(img_numpy)
            ids.append(id)

        except Exception as e:
            continue

    return faceSamples, ids

print("\nTraining data Processing...")
faces, ids = getImagesAndLabels(dataset_path)

print(f"\nModel Train Processing (Unique Users: {len(set(ids))})...")
recognizer.train(faces, np.array(ids))

# Model Save
recognizer.save(model_save_path)

print(f"\n Training Successfully!")
print(f"Model save : '{model_save_path}'")


Training data Processing...
Training Images 2032 Reading

Model Train Processing (Unique Users: 116)...

 Training Successfully!
Model save : 'trainer.yml'


## **5. Model Download**

In [ ]:
from google.colab import files

print("Files Download...")
files.download('trainer.yml')
files.download('Final_Attendance_Dataset/names.txt') # ID list

print(f"\n Model Downlaod Successfully!")

Files Download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Model Downlaod Successfully!


## **6. Accuracy Check**

In [6]:
import cv2
import os
import numpy as np
from PIL import Image

# 1. Files Path
test_path = 'Final_Attendance_Dataset/test' # Test folder
model_path = 'trainer.yml'                  # Train Model

# 2. Model Load
if not os.path.exists(model_path):
    print(f"Error: '{model_path}' Not Found File")
else:
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read(model_path)

    # Face Detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    correct_predictions = 0
    total_images = 0

    print("Testing Processing...")

    # Check Test Folder
    files = [f for f in os.listdir(test_path) if f.endswith(('.jpg', '.png'))]

    for file in files:
        total_images += 1
        img_path = os.path.join(test_path, file)

        # 1. ID (Actual ID)
        # Format : User_1_5.jpg -> ID 1
        try:
            actual_id = int(file.split("_")[1])
        except:
            continue

        # 2. Picture Read
        pil_img = Image.open(img_path).convert('L')
        img_numpy = np.array(pil_img, 'uint8')

        # 3. Model (Prediction)
        faces = face_cascade.detectMultiScale(img_numpy, 1.2, 5)

        predicted_id = -1
        confidence = -1

        if len(faces) > 0:
            for (x, y, w, h) in faces:
                predicted_id, confidence = recognizer.predict(img_numpy[y:y+h, x:x+w])
                break
        else:
            predicted_id, confidence = recognizer.predict(img_numpy)

        # 4. check Correct,Incorrect
        if predicted_id == actual_id:
            correct_predictions += 1

        # Show Results
        if total_images % 100 == 0:
            print(f"Processed: {total_images} | Current Accuracy: {round((correct_predictions/total_images)*100, 2)}%")

    # Final Output
    if total_images > 0:
        accuracy = (correct_predictions / total_images) * 100
        print("\n" + "="*30)
        print(f"Total testing images: {total_images}")
        print(f"No. of Correct Predictions: {correct_predictions}")
        print(f"Accuracy: {round(accuracy, 2)}%")
        print("="*30)
    else:
        print("Test Folder Empty")

Testing Processing...
Processed: 100 | Current Accuracy: 88.0%
Processed: 200 | Current Accuracy: 87.5%
Processed: 300 | Current Accuracy: 86.67%
Processed: 400 | Current Accuracy: 86.0%
Processed: 500 | Current Accuracy: 85.0%

Total testing images: 537
No. of Correct Predictions: 455
Accuracy: 84.73%
